## 01 Lcel Basics

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_template("Tell me a short fact about {topic}")
llm = ChatGroq(model='llama-3.3-70b-versatile')
parser = StrOutputParser()

# LCEL Magic: Pipe the components together
chain = prompt | llm | parser

result = chain.invoke({"topic": "space"})
print(result)


## 02 Routing Chains

In [ ]:
from langchain_core.runnables import RunnableBranch

# Example condition function (Must be a function for RunnableBranch)
def is_math(x: dict) -> bool:
    return "math" in x["topic"].lower()

math_chain = ChatPromptTemplate.from_template("Solve this math problem: {topic}") | llm | parser
general_chain = ChatPromptTemplate.from_template("Answer this general question: {topic}") | llm | parser

branch = RunnableBranch(
    (is_math, math_chain),
    general_chain
)

print(branch.invoke({"topic": "math: 2+2"}))


## 03 Fallbacks And Retries

In [ ]:
primary_llm = ChatGroq(model='llama-3.3-70b-versatile', max_retries=3)
fallback_llm = ChatGroq(model='llama3-8b-8192')

# If the primary LLM fails (e.g., rate limit), it automatically uses the fallback
resilient_llm = primary_llm.with_fallbacks([fallback_llm])

chain = prompt | resilient_llm | parser
print(chain.invoke({"topic": "ocean"}))
